# Face Recognition – Feature Extraction (Embeddings)

## Objective
This notebook extracts deep facial embeddings from detected face images using a CNN-based model.
The generated embeddings are used to build a feature database that supports face matching,
classification, and attendance logging.


## Required Libraries
Import packages for numerical computation, file handling, and deep face models.

In [7]:
import os
import pickle
import numpy as np
from tqdm import tqdm
from deepface import DeepFace

## Dataset Paths
Define input and output directories used for embedding extraction and database storage.
``

In [8]:
# Faces cropped by face_detection module
CROPPED_FACES_DIR = r"D:\face-recognition-attendance\outputs\cropped_faces"
DB_DIR = r"D:\face-recognition-attendance\outputs\database"

DB_PATH = os.path.join(DB_DIR, "face_embeddings.pkl")

os.makedirs(DB_DIR, exist_ok=True)

## Face Embedding Model
A CNN-based FaceNet model is used to extract robust and discriminative facial embeddings.
Each face image is converted into a fixed-length numerical vector.

In [9]:
def extract_embedding(img_path, model_name="Facenet512"):
    """
    Extract 512-D face embedding using FaceNet (CNN).
    """
    rep = DeepFace.represent(
        img_path=img_path,
        model_name=model_name,
        enforce_detection=False
    )
    return np.array(rep[0]["embedding"])

## Embedding Extraction Function
This function takes a face image path as input and returns its corresponding face embedding vector.


In [10]:
database = {}

subjects = sorted(os.listdir(CROPPED_FACES_DIR))

for subject in tqdm(subjects):
    subject_path = os.path.join(CROPPED_FACES_DIR, subject)
    if not os.path.isdir(subject_path):
        continue

    embeddings = []
    for img_name in os.listdir(subject_path):
        img_path = os.path.join(subject_path, img_name)
        emb = extract_embedding(img_path)
        embeddings.append(emb)

    # Average embedding for identity (best practice)
    database[subject] = {
            "mean": np.mean(embeddings, axis=0),
            "all": embeddings
        }

100%|██████████| 40/40 [06:30<00:00,  9.75s/it]


## Building the Feature Database
For each subject, embeddings are extracted from all available face images.
The mean embedding is computed to represent the identity in the database.


In [11]:
print(f"Total identities: {len(database)}")
sample_key = list(database.keys())[0]
print(sample_key, database[sample_key]["mean"].shape)

Total identities: 40
s1 (512,)


## Saving the Feature Database
Save the generated face embeddings database in Pickle format for later use
by matching and classification modules.


In [12]:
with open(DB_PATH, "wb") as f:
    pickle.dump(database, f)

print(f"Embedding database saved at: {DB_PATH}")

Embedding database saved at: D:\face-recognition-attendance\outputs\database\face_embeddings.pkl


## Summary
In this notebook, deep facial embeddings were extracted using a CNN-based model.
A feature database was successfully built and saved to support face recognition
and attendance tracking in the complete system pipeline.
